# 텍스트 임베딩 & 의미 유사도 (Sentence-Transformers) — Disaster Tweets — Colab

문장을 **벡터(임베딩)** 로 바꿔 **의미 유사도(코사인)** 로 비교·검색·분류하는 NLP 기본 예제입니다.

- 핵심 흐름: 사전학습 문장 임베딩 모델 → 문장 벡터 → **코사인 유사도** → 유사문장 검색 + 임베딩 기반 분류.
- IOAI 2025 **Chameleon / Concepts**(아이콘 설명↔비밀단어 의미 매칭) 의 기반 기법입니다. (공식 해법: `SentenceTransformer`+cosine)
  우리 12번 BERT는 *분류*, 11번 CLIP은 *이미지-텍스트* — 본 노트북은 *순수 텍스트 임베딩/의미유사도*.
- [NLP Getting Started](https://www.kaggle.com/c/nlp-getting-started) 트윗 사용. 토큰이 **노트북에 내장**되어 바로 실행됩니다.

> ⚙️ GPU 권장(작아서 CPU도 가능). ⚠️ 노트북에 개인 API 토큰이 평문으로 들어 있습니다. 외부 공유 금지.

## 0. Setup — 라이브러리 설치 & Kaggle 자격증명

In [ ]:
import sys, subprocess
for pkg in ["kaggle", "sentence-transformers", "scikit-learn", "numpy", "pandas"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)
print("라이브러리 준비 완료")

In [ ]:
import os
os.environ["KAGGLE_API_TOKEN"] = "KGAT_5dd261fded8e0d7eb2c29d8d65dfabea"
print("Kaggle 자격증명 설정 완료 (내장 토큰)")

## 1. Kaggle에서 트윗 데이터 다운로드

In [ ]:
import os, glob, zipfile
WORK_DIR = "/content" if os.path.isdir("/content") else os.getcwd()
from kaggle.api.kaggle_api_extended import KaggleApi
api = KaggleApi(); api.authenticate()
api.competition_download_files("nlp-getting-started", path=WORK_DIR, quiet=False)
for zp in glob.glob(os.path.join(WORK_DIR, "*.zip")):
    with zipfile.ZipFile(zp) as zf: zf.extractall(WORK_DIR)
    os.remove(zp)
print("다운로드 완료:", [f for f in sorted(os.listdir(WORK_DIR)) if f.endswith(".csv")])

## 2. 문장 임베딩 계산

사전학습 모델 `all-MiniLM-L6-v2` 로 각 트윗을 384차원 벡터로 인코딩합니다.

In [ ]:
import pandas as pd, numpy as np
from sentence_transformers import SentenceTransformer

df = pd.read_csv(os.path.join(WORK_DIR, "train.csv"))
texts = df["text"].fillna("").tolist()

model = SentenceTransformer("all-MiniLM-L6-v2")
emb = model.encode(texts, batch_size=64, show_progress_bar=True, normalize_embeddings=True)
print("임베딩 shape:", emb.shape)

## 3. 의미 유사도 검색 (코사인)

쿼리 문장과 가장 의미가 가까운 트윗들을 찾습니다.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def most_similar(query, k=5):
    q = model.encode([query], normalize_embeddings=True)
    sims = cosine_similarity(q, emb)[0]
    idx = sims.argsort()[::-1][:k]
    return [(round(float(sims[i]),3), texts[i][:90]) for i in idx]

for q in ["a huge fire is burning the forest", "I am so happy today"]:
    print("쿼리:", q)
    for s, t in most_similar(q): print(f"   {s}  {t}")
    print()

## 4. 임베딩 기반 분류

임베딩을 특징으로 간단한 로지스틱 회귀를 학습해, 임베딩이 의미 정보를 잘 담는지 확인합니다 (재난 트윗 분류).

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

y = df["target"].values
clf = LogisticRegression(max_iter=1000)
scores = cross_val_score(clf, emb, y, cv=5, scoring="f1")
print("임베딩 + LogReg  5-fold F1:", np.round(scores, 3), "| 평균", round(scores.mean(),3))

## 5. (참고) 임베딩 시각화 (PCA 2D)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
p2 = PCA(n_components=2).fit_transform(emb[:2000])
plt.figure(figsize=(6,5))
plt.scatter(p2[:,0], p2[:,1], c=y[:2000], cmap="coolwarm", s=6, alpha=.5)
plt.title("Tweet embeddings (PCA 2D) — red=disaster"); plt.show()

## 🏁 제출 안내

이 노트북은 **텍스트 임베딩/의미 유사도** 연습용 데모입니다(제출 리더보드 없음).

- 데이터 출처: **[NLP Getting Started](https://www.kaggle.com/c/nlp-getting-started)**

> IOAI 2025 *Chameleon / Concepts*(아이콘↔비밀단어 의미 매칭) 의 기반 기법(문장 임베딩 + 코사인 유사도)입니다.